# BARAM 2026 풍력 발전량 예측 EDA 및 전처리 파이프라인

상세 보고서 `docs/EDA.md`, 전처리 `preprocessing.py`, 평가 `evaluation.py`를 실행형으로 연결한 노트북입니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from evaluation import CAPACITY_KWH, TARGET_COLS, metric_report, report_frame
from preprocessing import PreprocessingConfig, build_feature_bundle, make_2024_holdout, make_target_data, save_bundle

ROOT = Path('.').resolve()
DATA_DIR = ROOT / 'data'
assert DATA_DIR.exists()
pd.set_option('display.max_columns', 100)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
print(ROOT)

## 1. 파일 인벤토리와 공식 평가

실제 발전량이 설비용량 10% 이상인 시간만 평가하며 오차율 6% 이하 4원, 6~8% 3원, 8% 초과 0원입니다.

In [ ]:
inventory=[]
for p in sorted(DATA_DIR.rglob('*.csv')):
    d=pd.read_csv(p,encoding='utf-8-sig')
    inventory.append({'file':str(p.relative_to(DATA_DIR)),'rows':len(d),'columns':d.shape[1],'MB':p.stat().st_size/1e6})
display(pd.DataFrame(inventory))

## 2. 라벨 EDA

In [ ]:
labels=pd.read_csv(DATA_DIR/'train'/'train_labels.csv',encoding='utf-8-sig',parse_dates=['kst_dtm']).set_index('kst_dtm').sort_index()
rows=[]
for c in TARGET_COLS:
    s=labels[c].dropna(); cap=CAPACITY_KWH[c]
    rows.append({'group':c,'valid':len(s),'missing':labels[c].isna().sum(),'zero':s.eq(0).sum(),'eval_rate_%':100*s.ge(.1*cap).mean(),'mean_cf_%':100*s.mean()/cap,'median_cf_%':100*s.median()/cap,'max':s.max(),'above_cap':s.gt(cap).sum()})
display(pd.DataFrame(rows).set_index('group'))
display(labels.corr())

In [ ]:
monthly=pd.DataFrame({c:(labels[c]/CAPACITY_KWH[c]).groupby(labels.index.month).mean() for c in TARGET_COLS})
hourly=pd.DataFrame({c:(labels[c]/CAPACITY_KWH[c]).groupby(labels.index.hour).mean() for c in TARGET_COLS})
fig,ax=plt.subplots(1,2,figsize=(15,4))
monthly.plot(marker='o',ax=ax[0],title='월별 평균 CF')
hourly.plot(marker='o',ax=ax[1],title='종료시간별 평균 CF')
plt.tight_layout(); plt.show()

## 3. 누수 방지형 전처리

동일 공개시각 예보 배치 안의 grid별 시간보간, Train grid 중앙값 fallback, U/V 풍속 파생, 공간 통계, 풍속 grid pivot, 시간 순환 특성을 생성합니다. `aggregate`, `wind_grid`, `hybrid` 모드를 지원합니다.

In [ ]:
CONFIG=PreprocessingConfig(mode='hybrid',aggregate_stats=('mean','std','min','max'),dtype='float32',drop_constant=True,add_missing_indicators=True)
bundle=build_feature_bundle(DATA_DIR,CONFIG)
X_train,X_test,y_train=bundle.X_train,bundle.X_test,bundle.y_train
print('X_train',X_train.shape,'X_test',X_test.shape,'y',y_train.shape)
display(pd.Series(bundle.diagnostics, name='value').to_frame())
assert X_train.columns.equals(X_test.columns)
assert X_train.isna().sum().sum()==0 and X_test.isna().sum().sum()==0
assert np.isfinite(X_train.to_numpy()).all() and np.isfinite(X_test.to_numpy()).all()

## 4. 타깃 전략과 2024 홀드아웃

`all`, 평가대상만 쓰는 `hard`, 저발전 가중치를 낮춘 `soft`, 고발전 가중치를 높인 `settlement`를 같은 시간 fold에서 비교합니다.

In [ ]:
out=[]
for c in TARGET_COLS:
    for strategy in ['all','hard','soft','settlement']:
        mask,ycf,w=make_target_data(y_train,c,strategy=strategy)
        out.append({'group':c,'strategy':strategy,'rows':mask.sum(),'weight_mean':w[mask].mean(),'target_cf_mean':ycf[mask].mean()})
display(pd.DataFrame(out))
fit_time,valid_time=make_2024_holdout(X_train.index)
print('train hours',fit_time.sum(),'valid hours',valid_time.sum())

## 5. 선택 실행: ExtraTrees 진단 모델

In [ ]:
RUN_MODEL=False
if RUN_MODEL:
    from sklearn.ensemble import ExtraTreesRegressor
    idx=X_train.index[valid_time]
    pred=pd.DataFrame(index=idx,columns=TARGET_COLS,dtype=float)
    for c in TARGET_COLS:
        usable,ycf,w=make_target_data(y_train,c,strategy='hard')
        mask=fit_time & usable.to_numpy()
        m=ExtraTreesRegressor(n_estimators=200,min_samples_leaf=2,max_features=.8,n_jobs=1,random_state=42)
        m.fit(X_train.loc[mask],ycf.loc[mask],sample_weight=w.loc[mask])
        pred[c]=np.clip(m.predict(X_train.loc[valid_time]),0,1.02)*CAPACITY_KWH[c]
    report=metric_report(y_train.loc[idx,TARGET_COLS],pred)
    print({k:report[k] for k in ['score','one_minus_nmae','ficr']})
    display(report_frame(report))
else:
    print('RUN_MODEL=False')

## 6. 저장과 CLI

노트북 없이 `python preprocessing.py --data-dir data --output-dir artifacts --mode hybrid`로 같은 파이프라인을 실행할 수 있습니다.

In [ ]:
SAVE_ARTIFACTS=False
if SAVE_ARTIFACTS:
    save_bundle(bundle,ROOT/'artifacts',CONFIG)
    print('저장 완료')

## 7. 제출 파일 생성 함수

In [ ]:
def make_submission(test_pred,output_path):
    sample=pd.read_csv(DATA_DIR/'sample_submission.csv',encoding='utf-8-sig',parse_dates=['forecast_kst_dtm'])
    pred=test_pred.copy(); pred.index=pd.to_datetime(pred.index)
    pred=pred.reindex(sample['forecast_kst_dtm'])
    if pred[TARGET_COLS].isna().any().any(): raise ValueError('누락 시간 또는 NaN')
    out=sample.copy()
    for c in TARGET_COLS:
        upper=21_150 if c=='kpx_group_3' else CAPACITY_KWH[c]
        out[c]=np.clip(pred[c].to_numpy(),0,upper)
    assert len(out)==8760 and np.isfinite(out[TARGET_COLS].to_numpy()).all()
    out.to_csv(output_path,index=False,encoding='utf-8-sig')
    return out

## 8. 누수 체크

- 동시/미래 SCADA 금지
- validation 라벨 기반 변환 금지
- Test 통계로 학습 전처리 fitting 금지
- 그룹 3의 2022 라벨 0 대체 금지
- metric 전 시간키 정렬 확인
- 제출 8,760행과 NaN/inf 확인

## 부록: 전체 EDA 보고서

In [ ]:
p=ROOT/'EDA.md'
display(Markdown(p.read_text(encoding='utf-8'))) if p.exists() else print('EDA.md 없음')